# IFC Semantic and Spatial Relationships — HouseR26_etmaglari

In [1]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Import the needed TopologicPy Classes

In [2]:
from topologicpy.Cluster import Cluster
from topologicpy.Color import Color
from topologicpy.Topology import Topology
from topologicpy.Graph import Graph
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper

c:\Users\etmaglari\IAAC\etmaglari_gML\.gmlenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Check the TopologicPy version

In [3]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.31) is OLDER than the latest version (0.9.33) from PyPI. Please consider upgrading to the latest version.


## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [4]:
renderer = "vscode"

## 4. Set default mappings (do not change key names)

In [5]:
# --- Semantic relationship colours ---
semantic_rels_color_mapping = {
    "IfcRelConnectsPathElements":        "#440154",
    "IfcRelContainedInSpatialStructure": "#31688E",
    "IfcRelFillsElement":                "#35B779",
    "IfcRelSpaceBoundary":               "#FDE725",
    "IfcRelVoidsElement":                "#E64B5D",
}

# --- Spatial relationship colours ---
spatial_rels_color_mapping = {
    "contains":     "#FF0000",
    "coveredBy":    "#0000C8",
    "covers":       "#0000C8",
    "crosses":      "#0098FF",
    "disjoint":     "#2CFF96",
    "equals":       "#97FF00",
    "overlaps":     "#FFEA00",
    "touches":      "#550E55",
    "within":       "#FF0000",
    "near":         "#AAAAAA",
    "intermediate": "#666666",
    "far":          "#000000",
}

# --- IFC element colours (shared by both parts) ---
ifc_color_mapping = {
    "ifcsite":             "#AAAAAA",
    "ifcbuilding":         "#888888",
    "ifcbuildingstorey":   "#FFFFFF",
    "ifcspace":            "#FDE725",
    "ifcwall":             "#F8765C",
    "ifcwallstandardcase": "#E64B5D",
    "ifccurtainwall":      "#482878",
    "ifcslab":             "#B4DE2C",
    "ifcroof":             "#E67E22",
    "ifcopeningelement":   "#35B779",
    "ifcdoor":             "#3E4989",
    "ifcwindow":           "#D41159",
}

## 5. Specify the path to the IFC file and what to include

In [6]:
ifc_file_path = r"C:\Users\etmaglari\IAAC\etmaglari_gML\Homework02\House.ifc"

## 6. Import the IFC file as graph

In [7]:
graph = Graph.ByIFCPath(ifc_file_path,
                        importMode="topology",
                        dictionaryMode="basic",
                        storeBREP=True,
                        includeTypes=["IfcWall", "IfcWallStandardCase", "IfcCurtainWall",
                                      "IfcSlab", "IfcRoof", "IfcOpeningElement",
                                      "IfcDoor", "IfcWindow",
                                      "IfcSpace", "IfcBuildingStorey",
                                      "IfcBuilding", "IfcSite"],
                        includeRels=["IfcRelContainedInSpatialStructure",
                                     "IfcRelVoidsElement",
                                     "IfcRelFillsElement",
                                     "IfcRelSpaceBoundary"],
                       )

## 7. Extract breps from the vertices and set colours for vertices and edges

In [8]:
vertices = Graph.Vertices(graph)
boxes = []
for v in vertices:
    d = Topology.Dictionary(v)
    brep = Dictionary.ValueAtKey(d, "BREP")
    if brep:
        topology = Topology.ByBREPString(brep)
        box = Topology.BoundingBox(topology)
        boxes.append(box)
    ifc_type = Dictionary.ValueAtKey(d, "IFC_type", "unknown")
    vertexColor = ifc_color_mapping.get(ifc_type.lower(), "white")
    d = Dictionary.SetValuesAtKeys(d, ["size", "color"], [10, vertexColor])
    v = Topology.SetDictionary(v, d)

edges = Graph.Edges(graph)
for e in edges:
    d = Topology.Dictionary(e)
    ifc_rel = Dictionary.ValueAtKey(d, "IFC_type")
    edgeColor = semantic_rels_color_mapping.get(ifc_rel, "white")
    d = Dictionary.SetValuesAtKeys(d, ["width", "color"], [3, edgeColor])
    e = Topology.SetDictionary(e, d)

## 8. Show the semantic graph

In [9]:
Topology.Show(boxes, graph,
              faceOpacity=0.1,
              sagitta=0.15,
              absolute=False,
              backgroundColor="black",
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              width=800,
              height=600,
              renderer=renderer)

## 9. Use Pyvis for an alternative visualisation

In [10]:
pyvis_graph = Graph.PyvisGraph(graph, path=r"C:\Users\etmaglari\Desktop\pyvis_graph.html",
                               vertexSizeKey="size",
                               vertexColorKey="color",
                               vertexLabelKey="IFC_type",
                               edgeWeightKey="width",
                               edgeColorKey="color")

C:\Users\etmaglari\Desktop\pyvis_graph.html


---
# Part B: Spatial Relationships
Geometric relationships between building elements (touches, overlaps, within) computed from bounding boxes.

## 10. Specify spatial relationship types and element types to include

In [11]:
includeTypes_spatial = ["ifcwall", "ifcwallstandardcase", "ifcslab", "ifcroof",
                        "ifcopeningelement", "ifcdoor"]

includeRels_spatial = ["overlaps", "touches"]

## 11. Import IFC objects and create bounding boxes

In [12]:
ifc_objects = Topology.ByIFCPath(ifc_file_path,
                                 includeTypes=includeTypes_spatial,
                                 dictionaryMode="basic")
spatial_boxes = [Topology.BoundingBox(obj) for obj in ifc_objects]
print(f"Imported {len(ifc_objects)} IFC objects.")

IFCFastTopology.Parse - Parsed 446868 entities in 5.008s.
IFCFastTopology.TopologiesByEntities - Created 114 topologies; skipped 3 products in 166.527s.
Imported 114 IFC objects.


## 12. Show the simplified IFC model

In [13]:
Topology.Show(spatial_boxes,
              faceOpacity=0.2,
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

## 13. Create the spatial relationships graph

In [14]:
spatial_graph = Graph.BySpatialRelationships(spatial_boxes, include=includeRels_spatial)
print(f"Created a graph with {len(Graph.Vertices(spatial_graph))} vertices and {len(Graph.Edges(spatial_graph))} edges.")

Created a graph with 112 vertices and 344 edges.


## 14. Map edge and vertex colours

In [15]:
spatial_edges = Graph.Edges(spatial_graph)
for e in spatial_edges:
    d = Topology.Dictionary(e)
    relFwd = Dictionary.ValueAtKey(d, "relFwd")
    edgeColor = spatial_rels_color_mapping.get(relFwd, "white")
    d = Dictionary.SetValuesAtKeys(d, ["color", "width"], [edgeColor, 3])
    e = Topology.SetDictionary(e, d)

spatial_vertices = Graph.Vertices(spatial_graph)
for v in spatial_vertices:
    d = Topology.Dictionary(v)
    ifc_type = Dictionary.ValueAtKey(d, "IFC_type", "unknown")
    vertexColor = ifc_color_mapping.get(ifc_type.lower(), "white")
    d = Dictionary.SetValuesAtKeys(d, ["size", "color"], [10, vertexColor])
    v = Topology.SetDictionary(v, d)

## 15. Show the spatial relationships result

In [16]:
Topology.Show(spatial_boxes, spatial_graph,
              faceOpacity=0.1,
              sagitta=0.15,
              absolute=False,
              backgroundColor="black",
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              width=800,
              height=600,
              renderer=renderer)

## 16. Use Pyvis for an alternative visualisation

In [17]:
pyvis_spatial = Graph.PyvisGraph(spatial_graph,
                                 path=r"C:\Users\etmaglari\Desktop\pyvis_graph_spatial.html",
                                 vertexSizeKey="size",
                                 vertexColorKey="color",
                                 vertexLabelKey="IFC_type",
                                 edgeWeightKey="width",
                                 edgeColorKey="color")

C:\Users\etmaglari\Desktop\pyvis_graph_spatial.html
